In [7]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
import os
from openai import OpenAI


client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

class AgentStateLLM(TypedDict):
    messages: Annotated[list, operator.add]
    next_action: str
    iteration_count: int



In [8]:
def llm_tool(query: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o",
        max_token=150,
        messages=[{"role":"user", "content": f"Answer this query briefly: {query}"}]
    )

    return response.choices[0].message.content.strip()


In [9]:
# LLM reasoning

def reasoning_node_llm(state: AgentStateLLM):
    iteration_count = state.get("iterations_count",0)
    if iteration_count >= 3:
        return {
            "messages": ["Thought: I have gathered enough information"],
            "next_action": "end", "iteration_count": iteration_count
        }
    
    history = "\n".join(state["messages"])
    prompt = f""" You are an AI Agent answering: "Tell me about Tokyo and Japan"

Conversation so far:
{history}

Queries completed: {iteration_count}/3

You MUST make exactly 3 queries to gather information.
Respond ONLY with: QUERY: <your specific question>

Do NOT be conversational. Do not thank the user. Only output: QUERY: <question>"""
    
    decision = client.chat.completions.create(
        model="gpt-4o", max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    ).choices[0].message.content.strip()

    if decision.startswith("QUERY:"):
        return {"messages": [f"Thought: {decision}"], "next_action": "action",
                "iteration_count": iteration_count}
    return {"messages": [f"Thought: {decision}"], "next_action": "end",
            "iteration_count":iteration_count}


In [10]:
"""
1. what questions it's trying to answer
2. what info has been gatherd so far
3. how many queries it's allowed to make
4. Exactly how to format its response

"""

"\n1. what questions it's trying to answer\n2. what info has been gatherd so far\n3. how many queries it's allowed to make\n4. Exactly how to format its response\n\n"

In [11]:
## Action Exection

def action_node_llm(state: AgentStateLLM):
    last_thought = state["messages"][-1]
    query = last_thought.replace("Though: QUERY:", "").strip()
    result = llm_tool(query)
    return {
        "messages": [f"Action: query('{query}')", f"Observation: {result}"],
        "next_action": "reasoning",
        "iteration_count": state.get("iteration_count",0)+1
    }

In [12]:
## Graph construction

workflow_llm = StateGraph(AgentStateLLM)
workflow_llm.add_node("reasoning",reasoning_node_llm)
workflow_llm.add_node("action", action_node_llm)

workflow_llm.set_entry_point("reasoning")
workflow_llm.add_conditional_edges("reasoning", lambda s: s["next_action"],
                                   {"action":"action", "end": END} )
workflow_llm.add_edge("action", "reasoning")

app_llm = workflow_llm.compile()

In [13]:
result_llm = app_llm.invoke({
    "messages": ["User: Tell me about Tokyo and Japan"],
    "next_action": "",
    "iteration_count": 0
})

AuthenticationError: Error code: 401 - {'error': {'message': "You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the password field (with blank username) if you're accessing the API from your browser and are prompted for a username and password. You can obtain an API key from https://platform.openai.com/account/api-keys.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [14]:
print("\n=== ReAct Flow ===")
for msg in result_llm["messages"]:
    print(msg)


=== ReAct Flow ===


NameError: name 'result_llm' is not defined